In [45]:
import os
#os.chdir("../")
%pwd

'/home/abood/Desktop/ML_Model_Monitoring_System_for_Credit_Risk'

In [46]:
from pathlib import Path
from dataclasses import dataclass

In [47]:
@dataclass
class ModelTrainerConfig:
    root_dir:Path
    train_data_path: Path
    test_data_path: Path
    model_path: Path
    alpha: float 
    l1_ratio: float  
    target_column: str

In [48]:
from src.creditrisk.constants import *
from src.creditrisk.utils import read_yaml, create_directories

In [49]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_path=Path(config.root_dir) / "model.pkl",
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name
        )
        return model_trainer_config

In [50]:
import os
import joblib
from src.creditrisk.utils import logger
from sklearn.linear_model import ElasticNet
import pandas as pd

In [51]:
class ModelTrainer:
    def __init__(self , config: ModelTrainerConfig):
        self.config = config
    def train_model(self):
        # Read X and y files separately
        train_x = pd.read_csv(self.config.train_data_path.replace('train.csv', 'X_train.csv'))
        test_x = pd.read_csv(self.config.test_data_path.replace('test.csv', 'X_test.csv'))
        train_y = pd.read_csv(self.config.train_data_path.replace('train.csv', 'y_train.csv'))
        test_y = pd.read_csv(self.config.test_data_path.replace('test.csv', 'y_test.csv'))

        model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio)
        model.fit(train_x, train_y.values.ravel())

        joblib.dump(model, self.config.model_path)

In [52]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_traier = ModelTrainer(config=model_trainer_config)
    model_traier.train_model()
except Exception as e:
    logger.exception(e) 

[2026-05-01 22:40:11,306: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-05-01 22:40:11,310: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-01 22:40:11,315: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-05-01 22:40:11,317: INFO: common]: created directory at: artifacts
[2026-05-01 22:40:11,319: INFO: common]: created directory at: artifacts/model_trainer
